# Hướng dẫn chạy OmniVoice trên Google Colab

Chỉ cần bấm **Runtime -> Run all** (hoặc Ctrl + F9) để chạy từ trên xuống dưới.

Colab sẽ tự động:
1. Cài đặt các thư viện cần thiết & Cloudflare Tunnel siêu tốc
2. Tạo file demo_vn.py
3. Chạy Server và cung cấp đường link Cloudflare (https://xxxx.trycloudflare.com) cùng link Gradio dự phòng.

In [ ]:
!pip install -q --upgrade transformers>=5.3
!pip install -q omnivoice pydub gradio==6.11.0
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared

In [ ]:
%%writefile demo_vn.py
#!/usr/bin/env python3
# Copyright    2026  Xiaomi Corp.        (authors:  Han Zhu)
#
# See ../../LICENSE for clarification regarding multiple authors
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.
"""
Gradio demo for OmniVoice.

Supports voice cloning and voice design.

Usage:
    omnivoice-demo --model /path/to/checkpoint --port 8000
"""

import argparse
import logging
from typing import Any, Dict

import gradio as gr
import numpy as np
import torch

from omnivoice import OmniVoice, OmniVoiceGenerationConfig
from omnivoice.utils.common import get_best_device
from omnivoice.utils.lang_map import LANG_NAMES, lang_display_name


# ---------------------------------------------------------------------------
# Language list — all 600+ supported languages
# ---------------------------------------------------------------------------
_ALL_LANGUAGES = ["Auto"] + sorted(lang_display_name(n) for n in LANG_NAMES)


# ---------------------------------------------------------------------------
# Voice Design instruction templates
# ---------------------------------------------------------------------------
# Each option is displayed as "English / 中文".
# The model expects English for accents and Chinese for dialects.
_CATEGORIES = {
    "Giới tính": ["Nam", "Nữ"],
    "Độ tuổi": [
        "Trẻ em",
        "Thiếu niên",
        "Thanh niên",
        "Trung niên",
        "Người cao tuổi",
    ],
    "Độ cao giọng (Pitch)": [
        "Rất trầm",
        "Trầm",
        "Bình thường",
        "Cao",
        "Rất cao",
    ],
    "Phong cách": ["Nói thầm"],
    "Khẩu âm Tiếng Anh (Chỉ dùng cho tiếng Anh)": [
        "Giọng Mỹ",
        "Giọng Úc",
        "Giọng Anh",
        "Giọng Trung Quốc",
        "Giọng Canada",
        "Giọng Ấn Độ",
        "Giọng Hàn Quốc",
        "Giọng Bồ Đào Nha",
        "Giọng Nga",
        "Giọng Nhật",
    ],
    "Phương ngôn Tiếng Trung (Chỉ dùng cho tiếng Trung)": [
        "Tiếng Hà Nam",
        "Tiếng Thiểm Tây",
        "Tiếng Tứ Xuyên",
        "Tiếng Quý Châu",
        "Tiếng Vân Nam",
        "Tiếng Quế Lâm",
        "Tiếng Tế Nam",
        "Tiếng Thạch Gia Trang",
        "Tiếng Cam Túc",
        "Tiếng Ninh Hạ",
        "Tiếng Thanh Đảo",
        "Tiếng Đông Bắc",
    ],
}

_ATTR_INFO = {
    "Khẩu âm Tiếng Anh (Chỉ dùng cho tiếng Anh)": "Only effective for English speech.",
    "Phương ngôn Tiếng Trung (Chỉ dùng cho tiếng Trung)": "Only effective for Chinese speech.",
}

# ---------------------------------------------------------------------------
# Argument parser
# ---------------------------------------------------------------------------


def build_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(
        prog="omnivoice-demo",
        description="Launch a Gradio demo for OmniVoice.",
        formatter_class=argparse.RawTextHelpFormatter,
    )
    parser.add_argument(
        "--model",
        default="k2-fsa/OmniVoice",
        help="Model checkpoint path or HuggingFace repo id.",
    )
    parser.add_argument(
        "--device", default=None, help="Device to use. Auto-detected if not specified."
    )
    parser.add_argument("--ip", default="0.0.0.0", help="Server IP (default: 0.0.0.0).")
    parser.add_argument(
        "--port", type=int, default=7860, help="Server port (default: 7860)."
    )
    parser.add_argument(
        "--root-path",
        default=None,
        help="Root path for reverse proxy.",
    )
    parser.add_argument(
        "--share", action="store_true", default=False, help="Create public link."
    )
    parser.add_argument(
        "--no-asr",
        action="store_true",
        default=False,
        help="Skip loading Whisper ASR model. Reference text auto-transcription"
        " will be unavailable.",
    )
    parser.add_argument(
        "--asr-model",
        default="openai/whisper-large-v3-turbo",
        help="ASR model path or HuggingFace repo id"
        " (default: openai/whisper-large-v3-turbo).",
    )
    return parser


# ---------------------------------------------------------------------------
# Build demo
# ---------------------------------------------------------------------------


def build_demo(
    model: OmniVoice,
    checkpoint: str,
    generate_fn=None,
) -> gr.Blocks:
    sampling_rate = model.sampling_rate

    # -- shared generation core --
    def _gen_core(
        text,
        language,
        ref_audio,
        instruct,
        num_step,
        guidance_scale,
        denoise,
        speed,
        duration,
        preprocess_prompt,
        postprocess_output,
        mode,
        ref_text=None,
    ):
        if not text or not text.strip():
            return None, "Vui lòng nhập văn bản cần đọc."

        gen_config = OmniVoiceGenerationConfig(
            num_step=int(num_step or 32),
            guidance_scale=float(guidance_scale) if guidance_scale is not None else 2.0,
            denoise=bool(denoise) if denoise is not None else True,
            preprocess_prompt=bool(preprocess_prompt),
            postprocess_output=bool(postprocess_output),
        )

        lang = language if (language and language != "Auto") else None

        kw: Dict[str, Any] = dict(
            text=text.strip(), language=lang, generation_config=gen_config
        )

        if speed is not None and float(speed) != 1.0:
            kw["speed"] = float(speed)
        if duration is not None and float(duration) > 0:
            kw["duration"] = float(duration)

        if mode == "clone":
            if not ref_audio:
                return None, "Vui lòng tải lên một file âm thanh mẫu."
            kw["voice_clone_prompt"] = model.create_voice_clone_prompt(
                ref_audio=ref_audio,
                ref_text=ref_text,
            )

        if instruct and instruct.strip():
            kw["instruct"] = instruct.strip()

        try:
            audio = model.generate(**kw)
        except Exception as e:
            return None, f"Error: {type(e).__name__}: {e}"

        waveform = (audio[0] * 32767).astype(np.int16)
        return (sampling_rate, waveform), "Thành công."

    # Allow external wrappers (e.g. spaces.GPU for ZeroGPU Spaces)
    _gen = generate_fn if generate_fn is not None else _gen_core

    # =====================================================================
    # UI
    # =====================================================================
    theme = gr.themes.Soft(
        font=["Inter", "Arial", "sans-serif"],
    )
    css = """
    .gradio-container {max-width: 100% !important; font-size: 16px !important;}
    .gradio-container h1 {font-size: 1.5em !important;}
    .gradio-container .prose {font-size: 1.1em !important;}
    .compact-audio audio {height: 60px !important;}
    .compact-audio .waveform {min-height: 80px !important;}
    """

    # Reusable: language dropdown component
    def _lang_dropdown(label="Ngôn ngữ", value="Auto"):
        return gr.Dropdown(
            label=label,
            choices=_ALL_LANGUAGES,
            value=value,
            allow_custom_value=False,
            interactive=True,
            info="Để Auto để AI tự động nhận diện ngôn ngữ của bạn.",
        )

    # Reusable: optional generation settings accordion
    def _gen_settings():
        with gr.Accordion("Cài đặt nâng cao (Tùy chọn)", open=False):
            sp = gr.Slider(
                0.5,
                1.5,
                value=1.0,
                step=0.05,
                label="Tốc độ đọc",
                info="1.0 = Bình thường. >1 là Nhanh, <1 là Chậm. Sẽ bị bỏ qua nếu bạn đặt Độ dài ép buộc.",
            )
            du = gr.Number(
                value=None,
                label="Độ dài ép buộc (Giây)",
                info=(
                    "Để trống để dùng Tốc độ đọc. Điền số vào đây nếu muốn ép AI đọc đúng số giây đó."
                ),
            )
            ns = gr.Slider(
                4,
                64,
                value=32,
                step=1,
                label="Số bước lặp AI (Steps)",
                info="Mặc định 32. Càng thấp tốc độ tạo càng nhanh, Càng cao chất lượng giọng càng tốt.",
            )
            dn = gr.Checkbox(
                label="Khử tiếng ồn (Denoise)",
                value=True,
                info="Mặc định Bật. Bỏ tick để tắt tính năng khử ồn.",
            )
            gs = gr.Slider(
                0.0,
                4.0,
                value=2.0,
                step=0.1,
                label="Độ bám sát văn bản (CFG)",
                info="Mặc định: 2.0",
            )
            pp = gr.Checkbox(
                label="Tự xử lý âm thanh mẫu (Tiền xử lý)",
                value=True,
                info="apply silence removal and trimming to the reference "
                "audio, add punctuation in the end of reference text (if not already)",
            )
            po = gr.Checkbox(
                label="Tự xử lý kết quả (Hậu xử lý)",
                value=True,
                info="Tự động xóa các khoảng im lặng quá dài trong file âm thanh kết quả.",
            )
        return ns, gs, dn, sp, du, pp, po

    with gr.Blocks(theme=theme, css=css, title="OmniVoice - Trình tạo giọng nói Tiếng Việt") as demo:
        gr.Markdown(
            """
# OmniVoice Demo

State-of-the-art text-to-speech model for **600+ languages**, supporting:

- **Voice Clone** — Clone any voice from a reference audio
- **Voice Design** — Create custom voices with speaker attributes

Built with [OmniVoice](https://github.com/k2-fsa/OmniVoice)
by Xiaomi AI Lab Next-gen Kaldi team.
"""
        )

        with gr.Tabs():
            # ==============================================================
            # Voice Clone
            # ==============================================================
            with gr.TabItem("Nhái giọng (Clone)"):
                with gr.Row():
                    with gr.Column(scale=1):
                        vc_text = gr.Textbox(
                            label="Văn bản cần đọc (Nhập nội dung vào đây)",
                            lines=4,
                            placeholder="Nhập đoạn văn bản mà bạn muốn AI đọc thành tiếng vào đây...",
                        )
                        vc_ref_audio = gr.Audio(
                            label="Âm thanh mẫu (Upload file ghi âm)",
                            type="filepath",
                            elem_classes="compact-audio",
                        )
                        gr.Markdown(
                            "<span style='font-size:0.85em;color:#888;'>"
                            "Khuyên dùng: Tải lên file ghi âm dài từ 3 đến 10 giây. "
                            "</span>"
                        )
                        vc_ref_text = gr.Textbox(
                            label=("Văn bản của âm thanh mẫu (Nếu có)"),
                            lines=2,
                            placeholder="Transcript of the reference audio. Leave empty"
                            " to auto-transcribe via ASR models.",
                        )
                        vc_lang = _lang_dropdown("Ngôn ngữ")
                        with gr.Accordion("Mô tả thêm (Tùy chọn)", open=False):
                            vc_instruct = gr.Textbox(label="Instruct", lines=2)
                        (
                            vc_ns,
                            vc_gs,
                            vc_dn,
                            vc_sp,
                            vc_du,
                            vc_pp,
                            vc_po,
                        ) = _gen_settings()
                        vc_btn = gr.Button("🚀 BẮT ĐẦU TẠO GIỌNG NÓI", variant="primary")
                    with gr.Column(scale=1):
                        vc_audio = gr.Audio(
                            label="Kết quả âm thanh đầu ra",
                            type="numpy",
                        )
                        vc_status = gr.Textbox(label="Trạng thái hoạt động", lines=2)

                def _clone_fn(
                    text, lang, ref_aud, ref_text, instruct, ns, gs, dn, sp, du, pp, po
                ):
                    return _gen(
                        text,
                        lang,
                        ref_aud,
                        instruct,
                        ns,
                        gs,
                        dn,
                        sp,
                        du,
                        pp,
                        po,
                        mode="clone",
                        ref_text=ref_text or None,
                    )

                vc_btn.click(
                    _clone_fn,
                    inputs=[
                        vc_text,
                        vc_lang,
                        vc_ref_audio,
                        vc_ref_text,
                        vc_instruct,
                        vc_ns,
                        vc_gs,
                        vc_dn,
                        vc_sp,
                        vc_du,
                        vc_pp,
                        vc_po,
                    ],
                    outputs=[vc_audio, vc_status],
                )

            # ==============================================================
            # Voice Design
            # ==============================================================
            with gr.TabItem("Thiết kế giọng nói"):
                with gr.Row():
                    with gr.Column(scale=1):
                        vd_text = gr.Textbox(
                            label="Văn bản cần đọc (Nhập nội dung vào đây)",
                            lines=4,
                            placeholder="Nhập đoạn văn bản mà bạn muốn AI đọc thành tiếng vào đây...",
                        )
                        vd_lang = _lang_dropdown()

                        _AUTO = "Auto"
                        vd_groups = []
                        for _cat, _choices in _CATEGORIES.items():
                            vd_groups.append(
                                gr.Dropdown(
                                    label=_cat,
                                    choices=[_AUTO] + _choices,
                                    value=_AUTO,
                                    info=_ATTR_INFO.get(_cat),
                                )
                            )


                        (
                            vd_ns,
                            vd_gs,
                            vd_dn,
                            vd_sp,
                            vd_du,
                            vd_pp,
                            vd_po,
                        ) = _gen_settings()
                        vd_btn = gr.Button("🚀 BẮT ĐẦU TẠO GIỌNG NÓI", variant="primary")
                    with gr.Column(scale=1):
                        vd_audio = gr.Audio(
                            label="Kết quả âm thanh đầu ra",
                            type="numpy",
                        )
                        vd_status = gr.Textbox(label="Trạng thái hoạt động", lines=2)

                def _build_instruct(groups):
                    """Extract instruct text from UI dropdowns.

                    Language unification and validation is handled by
                    _resolve_instruct inside _preprocess_all.
                    """
                    selected = [g for g in groups if g and g != "Auto"]
                    if not selected:
                        return None
                    
                    vn_to_en = {
                        "Nam": "male", "Nữ": "female",
                        "Trẻ em": "child", "Thiếu niên": "teenager", "Thanh niên": "young adult", "Trung niên": "middle-aged", "Người cao tuổi": "elderly",
                        "Rất trầm": "very low pitch", "Trầm": "low pitch", "Bình thường": "moderate pitch", "Cao": "high pitch", "Rất cao": "very high pitch",
                        "Nói thầm": "whisper",
                        "Giọng Mỹ": "american accent", "Giọng Úc": "australian accent", "Giọng Anh": "british accent", "Giọng Trung Quốc": "chinese accent", "Giọng Canada": "canadian accent", "Giọng Ấn Độ": "indian accent", "Giọng Hàn Quốc": "korean accent", "Giọng Bồ Đào Nha": "portuguese accent", "Giọng Nga": "russian accent", "Giọng Nhật": "japanese accent",
                        "Tiếng Hà Nam": "河南话", "Tiếng Thiểm Tây": "陕西话", "Tiếng Tứ Xuyên": "四川话", "Tiếng Quý Châu": "贵州话", "Tiếng Vân Nam": "云南话", "Tiếng Quế Lâm": "桂林话", "Tiếng Tế Nam": "济南话", "Tiếng Thạch Gia Trang": "石家庄话", "Tiếng Cam Túc": "甘肃话", "Tiếng Ninh Hạ": "宁夏话", "Tiếng Thanh Đảo": "青岛话", "Tiếng Đông Bắc": "东北话"
                    }
                    
                    parts = []
                    for v in selected:
                        if v in vn_to_en:
                            parts.append(vn_to_en[v])
                        else:
                            if " / " in v:
                                en, zh = v.split(" / ", 1)
                                if "Dialect" in v.split(" / ")[0]:
                                    parts.append(zh.strip())
                                else:
                                    parts.append(en.strip())
                            else:
                                parts.append(v)
                    return ", ".join(parts)

                def _design_fn(text, lang, ns, gs, dn, sp, du, pp, po, *groups):
                    return _gen(
                        text,
                        lang,
                        None,
                        _build_instruct(groups),
                        ns,
                        gs,
                        dn,
                        sp,
                        du,
                        pp,
                        po,
                        mode="design",
                    )

                vd_btn.click(
                    _design_fn,
                    inputs=[
                        vd_text,
                        vd_lang,
                        vd_ns,
                        vd_gs,
                        vd_dn,
                        vd_sp,
                        vd_du,
                        vd_pp,
                        vd_po,
                    ]
                    + vd_groups,
                    outputs=[vd_audio, vd_status],
                )

    return demo


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------


def main(argv=None) -> int:
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s %(name)s %(levelname)s: %(message)s",
    )
    parser = build_parser()
    args = parser.parse_args(argv)

    device = args.device or get_best_device()

    checkpoint = args.model
    if not checkpoint:
        parser.print_help()
        return 0
    logging.info(f"Loading model from {checkpoint}, device={device} ...")
    model = OmniVoice.from_pretrained(
        checkpoint,
        device_map=device,
        dtype=torch.float16,
        load_asr=not args.no_asr,
        asr_model_name=args.asr_model,
    )
    print("Model loaded.")

    demo = build_demo(model, checkpoint)

    # Tự động chạy Cloudflare Tunnel song song để có đường truyền tốc độ cao (không bị bóp mạng)
    import subprocess, threading, re

    def _start_cloudflare(port=7860):
        try:
            cmd = ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{port}"]
            proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
            for line in iter(proc.stdout.readline, ''):
                m = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
                if m:
                    cf_url = m.group(0)
                    print("\n" + "=" * 70)
                    print("🚀 LINK CLOUDFLARE SIÊU TỐC (Dùng link này cho cả FPT & Viettel):")
                    print(f"👉 {cf_url}")
                    print("=" * 70 + "\n")
                    break
        except Exception as e:
            print(f"[Cloudflare Tunnel] Lỗi: {e}")

    threading.Thread(target=_start_cloudflare, args=(args.port or 7860,), daemon=True).start()

    demo.queue().launch(
        server_name=args.ip,
        server_port=args.port,
        share=args.share,
        root_path=args.root_path,
    )
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
!python demo_vn.py --share